# 📓 Caderno de Exercícios — Aula 02: Limpeza de Texto, Funções e Cruzamento Lógico
**Capacitação de Python para Ciência de Dados — Case Campe Supply**

> 💡 **Dica de Ouro:** Nesta aula vamos limpar as 6 colunas de texto (`setor`, `produto`, `vendedor`, `regiao`, `estado`, `cliente`). Existem várias formas de resolver: usando `difflib.get_close_matches`, usando distância de strings, expressões regulares (Regex) ou dicionários de mapeamento. Escolha ou combine a estratégia que preferir!

In [1]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

# === Checkpoint Inicial (Trazendo os dados limpos da Aula 01) ===
df = pd.read_excel('../Campe_Supply_Corrompido.xlsx', sheet_name='Campe Supply')
df = df.drop(columns=['UNNAMED: 0', 'coluna_inutil_0', 'coluna_inutil_1'])
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('ã', 'a')
df = df.drop_duplicates().reset_index(drop=True)
print('Base carregada da Aula 01:', df.shape)

df.head()

Base carregada da Aula 01: (1655, 10)


,data_da_venda,setor,produto,vendedor,regiao,estado,cliente,faturamento,margem_de_lucro,lucro
0,2035-01-01,Carnes,kuína,Amaral,Sudeste,Minas Gerais,Atacado,633.45,0.13,82.3485
1,2035-01-01,Latícinio,Queijo,Leo,Sudeste,Minas Gerais,Atacarejo,611.4,0.07,42.798
2,2035-01-01,Congelados,Sorvete,Leo,Nordeste,NaN,Atacarejo,721.52,0.08,NaN
3,2035-01-01,Carnes,Bovina,Daniele,Sudeste,São Paulo,Atacado,679.39,0.09,61.1451
4,2035-01-01,Carnes,Frango,Vanessa,Sul,Rio Grande do Sul,tarejo,964.52,0.19,183.2588


## 🎯 Missão 1: Investigando a Sujeira nas Categorias
Use `.value_counts()` ou `.unique()` nas 6 colunas de texto (`setor`, `produto`, `vendedor`, `regiao`, `estado`, `cliente`).

* **O que observar:** Quais são os nomes **verdadeiros** de cada categoria (aqueles que aparecem dezenas/centenas de vezes) e quais são os **erros de digitação** (aqueles que aparecem 1 a 4 vezes, como `"Cvrnes"`, `"Vacessa"`, `"tarejo"`)?

In [2]:
# 1. Explore os valores mais frequentes e os erros de digitação nas colunas de texto
# Dica: você pode fazer um loop for passando pelas 6 colunas de texto!
# Seu código aqui:

colunas_texto = ['setor', 'produto', 'vendedor', 'regiao', 'estado', 'cliente']

for coluna in colunas_texto:
    print(f'===== Coluna: {coluna.upper()} (Total de valores únicos: {df[coluna].nunique()})=====')
    print(df[coluna].value_counts().head(15))
    print('\n')

===== Coluna: SETOR (Total de valores únicos: 139)=====
setor
Congelados    367
Latícinio     364
Bebidas       353
Carnes        336
Bebidqs         2
Beyidas         2
Carncs          2
Bybidas         2
rarnes          2
Bebmdas         2
Cwrnes          2
Labícinio       2
Bpbidas         2
Carbes          2
Latícinso       2
Name: count, dtype: int64


===== Coluna: PRODUTO (Total de valores únicos: 152)=====
produto
Queijo           215
Sorvete          214
Frango           153
Refrigerantes    150
Cervejas         149
Bovina           138
Leite            124
Lasanha           74
Pizza             68
Suína             65
Sucos             56
Coalhada          20
Luite              2
Bhvina             2
Suwna              2
Name: count, dtype: int64


===== Coluna: VENDEDOR (Total de valores únicos: 152)=====
vendedor
Vanessa    433
Amaral     311
Leo        228
Daniele    213
Jonas      113
Evandro    108
Vanesta      3
aeo          3
Vacessa      3
qeo          3
Daniehe      

## 🎯 Missão 2: Criando uma Função para Corrigir Erros de Digitação
Temos mais de 140 variações de palavras digitadas com 1 letra errada. Fazer `.replace()` na mão levaria horas!

Crie uma lógica para corrigir automaticamente as palavras de cada coluna comparando-as com a lista de categorias oficiais:
* **Setores oficiais:** `['Congelados', 'Latícinio', 'Bebidas', 'Carnes']`
* **Produtos oficiais:** `['Queijo', 'Sorvete', 'Frango', 'Refrigerantes', 'Cervejas', 'Bovina', 'Leite', 'Lasanha', 'Pizza', 'Suína', 'Sucos', 'Coalhada']`
* **Vendedores oficiais:** `['Vanessa', 'Amaral', 'Leo', 'Daniele', 'Jonas', 'Evandro']`
* **Regiões oficiais:** `['Sudeste', 'Nordeste', 'Sul']`
* **Estados oficiais:** `['Rio de Janeiro', 'Minas Gerais', 'Bahia', 'Rio Grande do Norte', 'São Paulo', 'Sergipe', 'Rio Grande do Sul', 'Paraná', 'Santa Catarina']`
* **Clientes oficiais:** `['Atacarejo', 'Varejo', 'Atacado']`

* **Dica com `difflib`:** A função `get_close_matches(palavra, lista_oficial, n=1, cutoff=0.6)` retorna uma lista com a palavra mais parecida! Crie uma função `def` e aplique nas colunas usando `.apply()`.

In [3]:
# 2. Crie sua função de correção e aplique nas 6 colunas categóricas
# Seu código aqui:

dicionario_categorias = {
    'setor': ['Congelados', 'Latícinio', 'Bebidas', 'Carnes'],
    'produto': ['Queijo', 'Sorvete', 'Frango', 'Refrigerantes', 'Cervejas', 'Bovina', 'Leite', 'Lasanha', 'Pizza', 'Suína', 'Sucos', 'Coalhada'],
    'vendedor': ['Vanessa', 'Amaral', 'Leo', 'Daniele', 'Jonas', 'Evandro'],
    'regiao': ['Sudeste', 'Nordeste', 'Sul'],
    'estado': ['Rio de Janeiro', 'Minas Gerais', 'Bahia', 'Rio Grande do Norte', 'São Paulo', 'Sergipe', 'Rio Grande do Sul', 'Paraná', 'Santa Catarina'],
    'cliente': ['Atacarejo', 'Varejo', 'Atacado']
}

# Função de correção de palavras

def corrigir_palavras(valor, dicionario_categorias):
    if pd.isnull(valor):
        return np.nan
    
    palavra_limpa = str(valor).strip()
    match = get_close_matches(palavra_limpa, dicionario_categorias, n=1, cutoff=0.6)
    
    return match[0] if match else np.nan

In [4]:
for col, lista_valida in dicionario_categorias.items():
    df[col]=df[col].apply(lambda x: corrigir_palavras(x, lista_valida))
    print(f'Coluna {col} corrigida. Valores únicos restantes: {df[col].unique().tolist()}')

Coluna setor corrigida. Valores únicos restantes: ['Carnes', 'Latícinio', 'Congelados', 'Bebidas', nan]
Coluna produto corrigida. Valores únicos restantes: ['Suína', 'Queijo', 'Sorvete', 'Bovina', 'Frango', 'Coalhada', 'Cervejas', 'Sucos', nan, 'Leite', 'Pizza', 'Lasanha', 'Refrigerantes']
Coluna vendedor corrigida. Valores únicos restantes: ['Amaral', 'Leo', 'Daniele', 'Vanessa', nan, 'Evandro', 'Jonas']
Coluna regiao corrigida. Valores únicos restantes: ['Sudeste', 'Nordeste', 'Sul', nan]
Coluna estado corrigida. Valores únicos restantes: ['Minas Gerais', nan, 'São Paulo', 'Rio Grande do Sul', 'Rio Grande do Norte', 'Bahia', 'Paraná', 'Rio de Janeiro', 'Sergipe', 'Santa Catarina']
Coluna cliente corrigida. Valores únicos restantes: ['Atacado', 'Atacarejo', 'Varejo', nan]


## 🎯 Missão 3: Recuperação Inteligente de Nulos (Cruzamento Lógico)
Mesmo após corrigir a ortografia, ainda restam cerca de 83 a 85 valores vazios (`NaN`) em cada coluna.
Mas repare na regra de negócio da Campe Supply:
1. Todo `produto` pertence a um único `setor` (ex: *Queijo, Leite, Coalhada* $\rightarrow$ `Latícinio`; *Sorvete, Lasanha, Pizza* $\rightarrow$ `Congelados`; *Frango, Bovina, Suína* $\rightarrow$ `Carnes`; *Refrigerantes, Cervejas, Sucos* $\rightarrow$ `Bebidas`).
2. Todo `estado` pertence a uma única `regiao` (ex: *MG, SP, RJ* $\rightarrow$ `Sudeste`; *BA, RN, SE* $\rightarrow$ `Nordeste`; *RS, PR, SC* $\rightarrow$ `Sul`).

Use essa relação para preencher os valores nulos de `setor` (olhando para `produto`) e de `regiao` (olhando para `estado`)!

* **Dica:** Você pode usar dicionários com `.map()` combinados com `.fillna()`, ou usar `df.loc[...]` / `np.where()`.
* ✅ **Checkpoint de Validação:** Após o cruzamento, os valores nulos da coluna `setor` devem cair de **85 para apenas 4**, e os da coluna `regiao` devem cair de **83 para apenas 5**!

In [6]:
df.head()

,data_da_venda,setor,produto,vendedor,regiao,estado,cliente,faturamento,margem_de_lucro,lucro
0,2035-01-01,Carnes,Suína,Amaral,Sudeste,Minas Gerais,Atacado,633.45,0.13,82.3485
1,2035-01-01,Latícinio,Queijo,Leo,Sudeste,Minas Gerais,Atacarejo,611.4,0.07,42.798
2,2035-01-01,Congelados,Sorvete,Leo,Nordeste,NaN,Atacarejo,721.52,0.08,NaN
3,2035-01-01,Carnes,Bovina,Daniele,Sudeste,São Paulo,Atacado,679.39,0.09,61.1451
4,2035-01-01,Carnes,Frango,Vanessa,Sul,Rio Grande do Sul,Varejo,964.52,0.19,183.2588


In [7]:
# 3. Recupere os valores nulos de 'setor' (usando 'produto') e de 'regiao' (usando 'estado')
# Seu código aqui:
print('Valores nulos atualmente\n')
print(df[['setor', 'produto', 'regiao', 'estado']].isnull().sum())

Valores nulos atualmente

setor      85
produto    83
regiao     83
estado     83
dtype: int64


In [9]:
mapa_produto_setor = {
    'Queijo': 'Latícinio', 
    'Leite': 'Latícinio', 
    'Coalhada': 'Latícinio',
    'Sorvete': 'Congelados', 
    'Lasanha': 'Congelados', 
    'Pizza': 'Congelados',
    'Frango': 'Carnes', 
    'Bovina': 'Carnes', 
    'Suína': 'Carnes',
    'Refrigerantes': 'Bebidas', 'Cervejas': 'Bebidas', 'Sucos': 'Bebidas'
}

mapa_estado_regiao = {
    'Rio de Janeiro': 'Sudeste', 
    'Minas Gerais': 'Sudeste', 
    'São Paulo': 'Sudeste',
    'Bahia': 'Nordeste', 
    'Rio Grande do Norte': 'Nordeste', 
    'Sergipe': 'Nordeste',
    'Rio Grande do Sul': 'Sul', 
    'Paraná': 'Sul', 
    'Santa Catarina': 'Sul'
}

df['setor'] = df['setor'].fillna(df['produto'].map(mapa_produto_setor))
df['regiao'] = df['regiao'].fillna(df['estado'].map(mapa_estado_regiao))

print('Nulos depois do cruzamento lógico:\n', df[['setor', 'produto', 'regiao', 'estado']].isnull().sum())

Nulos depois do cruzamento lógico:
 setor       4
produto    83
regiao      5
estado     83
dtype: int64


In [10]:
df['produto'] = df['produto'].fillna('Não informado')
df['estado'] = df['estado'].fillna('Não informado')

In [11]:
print('Nulos depois do cruzamento lógico:\n', df[['setor', 'produto', 'regiao', 'estado']].isnull().sum())

Nulos depois do cruzamento lógico:
 setor      4
produto    0
regiao     5
estado     0
dtype: int64


In [12]:
df['setor'] = df['setor'].fillna('Não informado')
df['regiao'] = df['regiao'].fillna('Não informado')

In [13]:
print('Nulos depois do cruzamento lógico:\n', df[['setor', 'produto', 'regiao', 'estado']].isnull().sum())

Nulos depois do cruzamento lógico:
 setor      0
produto    0
regiao     0
estado     0
dtype: int64
